# 00 — Environment Check

Sanity check for the `retention-prediction` dev environment.

Run this notebook end-to-end on a fresh clone after `uv sync`. If all cells pass:
- The `retention` package is editable-installed.
- `set_global_seed()` + `configure_plot_style()` execute cleanly.
- Seaborn's colorblind palette renders correctly.
- Core ML / BigQuery / Jupyter deps import.
- BigQuery auth works against `pa-warehouse-prod` (or skips cleanly if SA key is absent).

If any cell fails, the dev environment isn't fully set up — fix before moving to Story 1+ work.

In [ ]:
# Cell 1 — Scaffolding (every notebook in this project starts with these two calls)
from retention import config

config.set_global_seed()
config.configure_plot_style()

print(f"SEED              = {config.SEED}")
print(f"PROJECT_ROOT      = {config.PROJECT_ROOT}")
print(f"BQ_PROJECT_ID     = {config.BQ_PROJECT_ID}")
print(f"BQ_DATASET_MARTS  = {config.BQ_DATASET_MARTS}")
print(f"BQ_TABLE_FEATURES = {config.BQ_TABLE_ATTRITION_FEATURES}")

In [ ]:
# Cell 2 — Verify reproducibility seed actually propagates
import random

import numpy as np

config.set_global_seed()  # re-seed for this cell
python_sample = [random.random() for _ in range(3)]
numpy_sample = np.random.rand(3).tolist()
print(f"random.random() x3  = {python_sample}")
print(f"np.random.rand() x3 = {numpy_sample}")
print("\nThese values are stable across runs because SEED is set.")
print("NOTE: hash() randomization is NOT reset here — set PYTHONHASHSEED=42 in the shell")
print("      before launching jupyter if you need deterministic hash(str) too.")

In [ ]:
# Cell 3 — Colorblind palette swatch (visual confirmation Story 0.2.9 landed)
import matplotlib.pyplot as plt
import seaborn as sns

palette = sns.color_palette(config.SEABORN_PALETTE)
fig, ax = plt.subplots(figsize=(8, 1.2))
for i, color in enumerate(palette):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=color))
    ax.text(i + 0.5, -0.25, str(i), ha="center", va="top", fontsize=9)
ax.set_xlim(0, len(palette))
ax.set_ylim(-0.5, 1)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"seaborn '{config.SEABORN_PALETTE}' palette — safe for red/green CB", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4 — Confirm key Loop 1 deps are importable (cheap insurance for fresh-clone reviewers)
import importlib

REQUIRED = [
    "pandas",
    "numpy",
    "sklearn",
    "xgboost",
    "google.cloud.bigquery",
    "pandas_gbq",
    "matplotlib",
    "seaborn",
    "mlflow",
    "shap",
    "survshap",
    "fairlearn",
]
missing = []
for name in REQUIRED:
    try:
        importlib.import_module(name)
    except ImportError as exc:
        missing.append((name, str(exc)))

if missing:
    print("MISSING:")
    for n, e in missing:
        print(f"  {n}: {e}")
else:
    print(f"All {len(REQUIRED)} core deps import cleanly. Environment is ready for Loop 1 modeling work.")

In [ ]:
# Cell 5 — BigQuery auth smoke (cost-capped at 1 MB; skips gracefully if SA key is absent)
#
# A clean import of `google.cloud.bigquery` (Cell 4) does NOT prove you can
# actually query the mart. This cell does — it runs `SELECT 1 AS smoke` against
# `pa-warehouse-prod` with a 1 MB byte cap, which is effectively free.
#
# Cross-platform path resolution: default is `~/.gcp/pa-warehouse-sa.json`
# (resolves to whichever user runs this on Windows / macOS / Linux), with
# `PA_WAREHOUSE_SA_KEY` env-var override. See `.env.example` for the contract.
#
# On a machine without GCP credentials, this cell prints a clear warning and
# returns — it does NOT raise, so `pytest --nbmake` still passes in CI.
import os
from pathlib import Path

_DEFAULT_SA_KEY = Path.home() / ".gcp" / "pa-warehouse-sa.json"
SA_KEY = Path(os.getenv("PA_WAREHOUSE_SA_KEY", str(_DEFAULT_SA_KEY)))

try:
    from google.cloud import bigquery
    from google.oauth2 import service_account

    if not SA_KEY.exists():
        print(f"⚠ SA key not found at {SA_KEY}.")
        print("  Set PA_WAREHOUSE_SA_KEY env var or place the SA JSON at the default path.")
        print("  Loop 1 modeling cells will fail until BigQuery auth is configured.")
    else:
        creds = service_account.Credentials.from_service_account_file(str(SA_KEY))
        client = bigquery.Client(project=config.BQ_PROJECT_ID, credentials=creds)
        job_config = bigquery.QueryJobConfig(maximum_bytes_billed=1 * 1024 * 1024)
        result = list(client.query("SELECT 1 AS smoke", job_config=job_config).result())
        print(f"BigQuery auth OK — SELECT 1 returned smoke={result[0].smoke}")
        print(f"  Authenticated as: {creds.service_account_email}")
        print(f"  Project:          {config.BQ_PROJECT_ID}")
except Exception as exc:
    print(f"⚠ BigQuery auth failed: {type(exc).__name__}: {exc}")
    print("  Loop 1 modeling cells will fail until this is fixed.")
